# Brent · `locality` (web: "Locality")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `locality` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "locality"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "locality"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["locality"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,35905,97.17
empty or blank,0,0.00
filled,1047,2.83
total,36952,100.00


### Findings · Step 1

Live API, 2026-11-10, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 35,905 | 97.17% |
| empty or blank | 0 | 0% |
| filled | 1,047 | 2.83% |

- **Mostly empty:** only 2.83% of records have a `locality`.

## Step 2 · Do the values match the schema type?

Count the Python type of every value in `df`: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [4]:
types = df["locality"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["locality"].value_counts().head(30).to_frame("count")

,count
locality,
NoneType,35905
str,1047


values that are not strings: 0


,count
locality,


### Findings · Step 2

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 1,047 |
| null | 35,905 |
| number | 0 |

- **Every filled value is a JSON string**, as the schema says (`string`). No type mismatch.

## Step 3 · Most common values

Count identical values of `locality` in `df` (exact text) and show the 30 most common.
`shown_as` uses `repr()` so hidden characters are visible.

In [5]:
print("distinct values:", df["locality"].nunique(), "| filled:", df["locality"].notna().sum())
top = df["locality"].value_counts().head(30).to_frame("count").reset_index()
top["shown_as"] = top["locality"].map(repr)
top

distinct values: 31 | filled: 1047


,locality,count,shown_as
0,Wembley,445,'Wembley'
1,London,275,'London'
2,"London,",117,"'London,'"
3,Harrow,72,'Harrow'
4,"Wembley,",36,"'Wembley,'"
5,Kilburn,15,'Kilburn'
6,Willesden,11,'Willesden'
7,Kingsbury,10,'Kingsbury'
8,Park Royal,10,'Park Royal'
9,Harlesden,8,'Harlesden'


### Findings · Step 3

Live API, 2026-11-10: 1,047 filled, 31 distinct.

- **Town or area names:** Wembley (445), London (275), Harrow (72), Kilburn, Willesden ...
- **The same place is written in several ways:** trailing commas (`London,` 117,
  `Wembley,` 36, `Harrow,` 4), a leading comma (`, Wembley`), case (`wembley`, `WEMBLEY`,
  `HARLESDEN`, `willesden`), and a cut-off value (`Wembl`).
- **Not places:** `BETWEEN` (1), and `OXGATE CENTRE` (1) is a site name.
- Some name places outside Brent (`North Kensington` 2, `Stanmore` 5, `Edgware` 6,
  `Harrow` 72); these are probably postal towns, not checked.